
# Competition Grade Traffic Demand Prediction

Features:
- Geohash spatial engineering
- Latitude / Longitude extraction
- Timestamp decomposition
- Cyclical encoding
- Missing value handling
- Target-safe preprocessing
- Optuna tuning for CatBoost
- 5-Fold CV
- CatBoost + LightGBM + XGBoost
- Stacking ensemble
- Submission generation


In [1]:

# Install if required
# !pip install catboost lightgbm xgboost optuna pygeohash -q


In [2]:

import pandas as pd
import numpy as np
import pygeohash as pgh
import optuna

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings("ignore")


In [3]:

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(train.shape)
print(test.shape)


(77299, 11)
(41778, 10)


In [4]:

def decode_geohash(g):

    try:
        lat, lon = pgh.decode(g)
        return pd.Series([lat, lon])
    except:
        return pd.Series([np.nan, np.nan])

train[['latitude','longitude']] = train['geohash'].apply(decode_geohash)
test[['latitude','longitude']] = test['geohash'].apply(decode_geohash)


In [5]:

def add_features(df):

    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['dayofweek'] = df['timestamp'].dt.dayofweek

    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)

    df['lat_lon_interaction'] = (
        df['latitude'] * df['longitude']
    )

    df['temp_lane_interaction'] = (
        df['Temperature'].fillna(0) *
        df['NumberofLanes']
    )

    return df

train = add_features(train)
test = add_features(test)


In [6]:
# Fill categorical columns
cat_cols = [
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]

for col in cat_cols:
    train[col] = train[col].fillna("Unknown")
    test[col] = test[col].fillna("Unknown")


# Fill numeric columns only
for col in train.columns:

    if pd.api.types.is_numeric_dtype(train[col]):

        median_value = train[col].median()

        train[col] = train[col].fillna(median_value)

        if col in test.columns:
            test[col] = test[col].fillna(median_value)

In [7]:

drop_cols = [
    'Index',
    'geohash',
    'timestamp'
]

X = train.drop(drop_cols + ['demand'], axis=1)
y = train['demand']

X_test = test.drop(drop_cols, axis=1)

for c in cat_cols:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

cat_idx = [X.columns.get_loc(c) for c in cat_cols]

print(X.shape)


(77299, 16)


In [8]:

def objective(trial):

    params = {

        'iterations': trial.suggest_int('iterations',500,2000),
        'depth': trial.suggest_int('depth',6,10),
        'learning_rate': trial.suggest_float('learning_rate',0.01,0.1),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg',1,10),
        'loss_function':'RMSE',
        'verbose':False,
        'random_seed':42
    }

    cv = KFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []

    for tr,va in cv.split(X):

        model = CatBoostRegressor(**params)

        model.fit(
            X.iloc[tr],
            y.iloc[tr],
            cat_features=cat_idx
        )

        pred = model.predict(X.iloc[va])

        scores.append(
            r2_score(y.iloc[va], pred)
        )

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

best_params = study.best_params
best_params['loss_function'] = 'RMSE'
best_params['verbose'] = False
best_params['random_seed'] = 42

print(best_params)


[I 2026-06-02 21:28:33,828] A new study created in memory with name: no-name-aa3a0423-de7b-4c02-bcbc-4e08e88c9b19
[I 2026-06-02 21:28:54,370] Trial 0 finished with value: 0.8863074007072561 and parameters: {'iterations': 1302, 'depth': 7, 'learning_rate': 0.03551534922560636, 'l2_leaf_reg': 6.744496789892828}. Best is trial 0 with value: 0.8863074007072561.
[I 2026-06-02 21:29:16,590] Trial 1 finished with value: 0.8443807743111561 and parameters: {'iterations': 1736, 'depth': 6, 'learning_rate': 0.012050276430556827, 'l2_leaf_reg': 7.740433562818676}. Best is trial 0 with value: 0.8863074007072561.
[I 2026-06-02 21:29:33,839] Trial 2 finished with value: 0.8744492389218181 and parameters: {'iterations': 1050, 'depth': 6, 'learning_rate': 0.03402329479372335, 'l2_leaf_reg': 1.0992524293531565}. Best is trial 0 with value: 0.8863074007072561.
[I 2026-06-02 21:30:14,024] Trial 3 finished with value: 0.9190152960145695 and parameters: {'iterations': 1612, 'depth': 7, 'learning_rate': 0.06

{'iterations': 1629, 'depth': 9, 'learning_rate': 0.07776493771239912, 'l2_leaf_reg': 3.070307522844684, 'loss_function': 'RMSE', 'verbose': False, 'random_seed': 42}


In [9]:

cat_model = CatBoostRegressor(**best_params)

lgb_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=10,
    random_state=42
)

xgb_model = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)


In [10]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_oof = np.zeros(len(X))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostRegressor(**best_params)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_idx,
        verbose=False
    )

    cat_oof[valid_idx] = model.predict(X_valid)

score = r2_score(y, cat_oof)

print("CatBoost OOF R2:", score)

CatBoost OOF R2: 0.9348452508200644


In [11]:

cat_model.fit(X,y,cat_features=cat_idx)

X_lgb = X.copy()
X_test_lgb = X_test.copy()

for c in cat_cols:
    X_lgb[c] = X_lgb[c].astype("category")
    X_test_lgb[c] = X_test_lgb[c].astype("category")

lgb_model.fit(X_lgb,y)

X_xgb = pd.get_dummies(X)
X_test_xgb = pd.get_dummies(X_test)

X_xgb, X_test_xgb = X_xgb.align(
    X_test_xgb,
    join='left',
    axis=1,
    fill_value=0
)

xgb_model.fit(X_xgb,y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001891 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 944
[LightGBM] [Info] Number of data points in the train set: 77299, number of used features: 16
[LightGBM] [Info] Start training from score 0.093942


,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [12]:

cat_pred_train = cat_model.predict(X)
lgb_pred_train = lgb_model.predict(X_lgb)
xgb_pred_train = xgb_model.predict(X_xgb)

stack_train = np.column_stack([
    cat_pred_train,
    lgb_pred_train,
    xgb_pred_train
])

meta_model = Ridge(alpha=1.0)
meta_model.fit(stack_train, y)


,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [13]:

cat_test = cat_model.predict(X_test)

lgb_test = lgb_model.predict(X_test_lgb)

xgb_test = xgb_model.predict(X_test_xgb)

stack_test = np.column_stack([
    cat_test,
    lgb_test,
    xgb_test
])

final_pred = meta_model.predict(stack_test)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_pred
})

submission.to_csv(
    'submission.csv',
    index=False
)

print(submission.shape)
submission.head()


(41778, 2)


,Index,demand
0,0,0.085773
1,1,0.040451
2,2,0.028321
3,3,0.017453
4,4,0.035729
